In [4]:
"""
Mtb Q-Loop Pharmacophore Model — Simple Version
================================================
What this does (and nothing more):
  1. Loads Q-loop compounds from Excel + their conformer SDF files
  2. Aligns every compound to a chosen template via MCS
  3. Extracts pharmacophore features (Donor, Acceptor, Aromatic,
     Hydrophobe, Cationic, Anionic) using RDKit ChemicalFeatures
  4. Clusters feature positions with DBSCAN to find consensus points
  5. Filters by coverage (fraction of molecules that carry the feature)
  6. Deduplicates spatially within each feature family
  7. Prints the final model and scores each molecule against it

No QSAR, no cross-validation, no Spearman, no enrichment factors.
"""

import os
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS, ChemicalFeatures
from sklearn.cluster import DBSCAN
from collections import defaultdict

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  — edit these to match your project
# ─────────────────────────────────────────────────────────────────────────────
CONFORMERS_DIR       = "Conformers"
TEMPLATE_NAME        = "CK_2_63.sdf"
EXCEL_PATH           = "Mtb.xlsx"

CLUSTERING_EPS       = 1.6   # Å  — DBSCAN neighbourhood radius
MIN_RADIUS           = 1.0   # Å  — smallest pharmacophore sphere
MAX_RADIUS           = 2.5   # Å  — largest pharmacophore sphere
MIN_FEATURE_COVERAGE = 0.50  # fraction of molecules that must carry a feature
INTRA_FAMILY_MIN_DIST= 2.0   # Å  — merge same-family points closer than this
ALIGNMENT_RMSD_CUTOFF= 5.0   # Å  — discard alignments worse than this

# ─────────────────────────────────────────────────────────────────────────────
# FEATURE DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
FDEF = """
DefineFeature Donor [$([N;!H0;v3,v4;+0,+1]),$(O[H]),$(S[H])]
  Family Donor
  Weights 1.0
EndFeature

DefineFeature Acceptor [$([N;H0;+0;v3]),$([O;H0;+0;v2]),$([F;$(F-[#6]);!$(FC[F,Cl,Br,I])]),$([S;H0;+0;v2])]
  Family Acceptor
  Weights 1.0
EndFeature

DefineFeature Aromatic [a]
  Family Aromatic
  Weights 1.0
EndFeature

DefineFeature Hydrophobe [$([c,s,br,I]),$([C;D3,D4;!$([C,N,O]=[C,N,O,S]);!$([CH2][O,N,S]);!$([CH][O,N,S]);!$([C][O,N,S])]),$([F,Cl,Br,I;$([F,Cl,Br,I]-[#6;!$([#6]~[#7,#8,#16])])])]
  Family Hydrophobe
  Weights 0.8
EndFeature

DefineFeature Cationic [$([NH2;+1]),$([NH3;+1]),$([NH4;+1]),$([nH;+1])]
  Family Cationic
  Weights 1.2
EndFeature

DefineFeature Anionic [$([C](=O)[O-]),$([P](=O)[O-]),$([S](=O)[O-]),$([c](=O)[O-])]
  Family Anionic
  Weights 1.2
EndFeature
"""

# ─────────────────────────────────────────────────────────────────────────────
# FILE / MOLECULE UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def find_sdf(name, directory):
    """Fuzzy-match a compound name to an SDF file (handles - / _ variants)."""
    target = str(name).lower().strip().replace('.sdf', '')
    variants = {target, target.replace('-', '_'), target.replace('_', '-')}
    for filename in os.listdir(directory):
        if filename.lower().endswith('.sdf'):
            stem = filename.lower().replace('.sdf', '')
            if stem in variants or target in stem:
                return os.path.join(directory, filename)
    return None


def load_mol(path, name=None):
    """
    Load an SDF that may contain multiple conformers.
    Returns a single RDKit Mol with all conformers attached, or None.
    """
    suppl = Chem.SDMolSupplier(path, removeHs=False, sanitize=False)
    mols  = [m for m in suppl if m is not None]
    if not mols:
        return None

    base = mols[0]
    try:
        Chem.SanitizeMol(base)
    except Exception:
        try:
            Chem.SanitizeMol(
                base,
                Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE
            )
        except Exception:
            return None

    for m in mols[1:]:
        try:
            base.AddConformer(m.GetConformer(), assignId=True)
        except Exception:
            continue

    if name:
        base.SetProp("_Name", name)
    return base


def align_to_template(mol, template):
    """
    Align mol to template using MCS atom mapping.
    Returns (mol, best_rmsd).  RMSD = 999 if alignment failed.
    """
    mcs = rdFMCS.FindMCS([template, mol], timeout=2)
    if mcs.numAtoms < 3:
        return mol, 999.0

    patt = Chem.MolFromSmarts(mcs.smartsString)
    if patt is None:
        return mol, 999.0

    probe_match    = mol.GetSubstructMatch(patt)
    template_match = template.GetSubstructMatch(patt)
    if not probe_match or not template_match:
        return mol, 999.0

    atom_map  = list(zip(probe_match, template_match))
    best_rmsd = 999.0

    for conf in mol.GetConformers():
        try:
            rmsd = rdMolAlign.AlignMol(
                mol, template,
                prbCid=conf.GetId(), refCid=0,
                atomMap=atom_map,
            )
            best_rmsd = min(best_rmsd, rmsd)
        except Exception:
            continue

    return mol, best_rmsd


# ─────────────────────────────────────────────────────────────────────────────
# PHARMACOPHORE MODEL BUILDING
# ─────────────────────────────────────────────────────────────────────────────

def build_pharmacophore(aligned_mols, factory):
    """
    Cluster feature positions across all aligned molecules and return
    a list of model points that pass the coverage threshold.

    Each model point is a dict:
        family    – feature family name (str)
        center    – 3-D centroid (np.array)
        radius    – sphere radius in Å (float)
        coverage  – fraction of input molecules that contribute (float)
    """
    n_mols   = len(aligned_mols)
    all_obs  = defaultdict(list)

    # Collect every feature position from every conformer of every molecule
    for mol_idx, mol in enumerate(aligned_mols):
        feats = factory.GetFeaturesForMol(mol)
        for conf in mol.GetConformers():
            cid = conf.GetId()
            for f in feats:
                all_obs[f.GetFamily()].append({
                    'pos':     np.array(f.GetPos(cid)),
                    'mol_idx': mol_idx,
                })

    print(f"\n  Feature families detected in dataset: {sorted(all_obs.keys())}")

    candidates = []

    for family, obs in all_obs.items():
        if len(obs) < 2:
            continue

        coords   = np.array([o['pos'] for o in obs])
        labels   = DBSCAN(eps=CLUSTERING_EPS, min_samples=2).fit_predict(coords)

        for label in set(labels):
            if label == -1:
                continue  # noise

            mask     = labels == label
            center   = coords[mask].mean(axis=0)
            dists    = np.linalg.norm(coords[mask] - center, axis=1)
            radius   = float(np.clip(np.mean(dists) * 1.5, MIN_RADIUS, MAX_RADIUS))
            u_mols   = len({obs[i]['mol_idx'] for i in np.where(mask)[0]})
            coverage = u_mols / n_mols

            candidates.append({
                'family':   family,
                'center':   center,
                'radius':   radius,
                'coverage': coverage,
            })

    # ── Filter by coverage ────────────────────────────────────────────────────
    kept = [p for p in candidates if p['coverage'] >= MIN_FEATURE_COVERAGE]
    dropped = [p for p in candidates if p['coverage'] < MIN_FEATURE_COVERAGE]
    for p in dropped:
        print(f"  Drop  {p['family']:<12}  coverage={p['coverage']:.0%}  "
              f"(threshold={MIN_FEATURE_COVERAGE:.0%})")

    # ── Intra-family spatial deduplication ────────────────────────────────────
    # Within each family keep only points separated by ≥ INTRA_FAMILY_MIN_DIST.
    # When two points are too close, keep the one with higher coverage.
    by_family = defaultdict(list)
    for p in kept:
        by_family[p['family']].append(p)

    final = []
    for family, pts in by_family.items():
        pts_sorted = sorted(pts, key=lambda x: -x['coverage'])
        accepted   = []
        for pt in pts_sorted:
            too_close = any(
                np.linalg.norm(pt['center'] - ex['center']) < INTRA_FAMILY_MIN_DIST
                for ex in accepted
            )
            if not too_close:
                accepted.append(pt)
            else:
                print(f"  Merge {family:<12}  redundant point within {INTRA_FAMILY_MIN_DIST}Å")
        final.extend(accepted)

    return final


# ─────────────────────────────────────────────────────────────────────────────
# SCORING  — how well does a molecule fit the model?
# ─────────────────────────────────────────────────────────────────────────────

def score_molecule(mol, model, factory):
    """
    For each model point find the nearest same-family feature across all
    conformers and compute a Gaussian-decay contribution:

        contribution = exp( -(d / radius)^2 )

    Returns:
        fit_score  – mean contribution across all model points (0–1)
        n_hits     – number of model points with d ≤ radius
    """
    if not model:
        return 0.0, 0

    contributions = []
    n_hits        = 0

    for pt in model:
        min_dist = np.inf
        for conf in mol.GetConformers():
            cid   = conf.GetId()
            feats = factory.GetFeaturesForMol(mol)
            for f in feats:
                if f.GetFamily() == pt['family']:
                    d = np.linalg.norm(np.array(f.GetPos(cid)) - pt['center'])
                    min_dist = min(min_dist, d)

        if min_dist == np.inf:
            contributions.append(0.0)
        else:
            contributions.append(np.exp(-(min_dist / pt['radius']) ** 2))
            if min_dist <= pt['radius']:
                n_hits += 1

    return float(np.mean(contributions)), n_hits


# ─────────────────────────────────────────────────────────────────────────────
# XML EXPORT
# ─────────────────────────────────────────────────────────────────────────────

def save_model_xml(model, path="pharmacophore_model.xml"):
    """
    Save the pharmacophore model as an XML file.

    Format is compatible with common pharmacophore viewers (LigandScout,
    Phase, PharmaGist) and can also be read back easily with Python's
    built-in xml.etree.ElementTree.

    Each feature point is written as:
        <feature id="1" family="Donor">
            <center x="4.15" y="0.14" z="0.18" />
            <sphere radius="2.12" />
            <coverage value="0.889" />
        </feature>
    """
    import xml.etree.ElementTree as ET
    from xml.dom import minidom

    root = ET.Element("pharmacophore")
    root.set("name",    "Mtb_Q-Loop")
    root.set("created", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
    root.set("n_features", str(len(model)))

    for i, pt in enumerate(model, 1):
        cx, cy, cz = pt['center']
        feat = ET.SubElement(root, "feature")
        feat.set("id",     str(i))
        feat.set("family", pt['family'])

        centre_el = ET.SubElement(feat, "center")
        centre_el.set("x", f"{cx:.4f}")
        centre_el.set("y", f"{cy:.4f}")
        centre_el.set("z", f"{cz:.4f}")

        sphere_el = ET.SubElement(feat, "sphere")
        sphere_el.set("radius", f"{pt['radius']:.4f}")

        cov_el = ET.SubElement(feat, "coverage")
        cov_el.set("value", f"{pt['coverage']:.4f}")

    # Pretty-print with indentation
    raw_xml  = ET.tostring(root, encoding="unicode")
    pretty   = minidom.parseString(raw_xml).toprettyxml(indent="  ")
    # minidom adds an extra XML declaration line — keep it, it's valid XML
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(pretty)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def run():
    print("=" * 70)
    print("Mtb Q-Loop Pharmacophore Model (simple)")
    print("=" * 70)

    # ── Feature factory ───────────────────────────────────────────────────────
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(FDEF)
    print("Feature factory built.")

    # ── Load data ─────────────────────────────────────────────────────────────
    df = pd.read_excel(EXCEL_PATH, sheet_name=1)
    df = df[df['Binding site'].str.upper() == 'Q-LOOP'].copy()

    ic50_col = pd.to_numeric(
        df['IC50 μM'].astype(str).str.replace('μM', '', regex=False),
        errors='coerce',
    )
    df['pIC50'] = -np.log10(ic50_col * 1e-6)
    df = df.dropna(subset=['pIC50']).reset_index(drop=True)
    print(f"Compounds loaded: {len(df)}  "
          f"(pIC50 {df['pIC50'].min():.2f} – {df['pIC50'].max():.2f})")

    # ── Load template ─────────────────────────────────────────────────────────
    t_path = find_sdf(TEMPLATE_NAME, CONFORMERS_DIR)
    if not t_path:
        raise FileNotFoundError(f"Template '{TEMPLATE_NAME}' not found in '{CONFORMERS_DIR}'")
    template = load_mol(t_path)
    if template is None:
        raise RuntimeError("Could not load template molecule")
    print(f"Template: {TEMPLATE_NAME}")

    # ── Align all molecules ───────────────────────────────────────────────────
    print("\nAligning compounds to template...")
    aligned_mols = []
    failed       = []

    for _, row in df.iterrows():
        path = find_sdf(row['Name'], CONFORMERS_DIR)
        if not path:
            print(f"  {row['Name']:<22}  NOT FOUND")
            failed.append(row['Name'])
            continue

        mol = load_mol(path, row['Name'])
        if mol is None:
            print(f"  {row['Name']:<22}  LOAD ERROR")
            failed.append(row['Name'])
            continue

        mol, rmsd = align_to_template(mol, template)
        if rmsd > ALIGNMENT_RMSD_CUTOFF:
            print(f"  {row['Name']:<22}  poor alignment  RMSD={rmsd:.2f} Å  (skipped)")
            failed.append(row['Name'])
            continue

        print(f"  {row['Name']:<22}  RMSD={rmsd:.2f} Å")
        aligned_mols.append(mol)

    print(f"\nAligned: {len(aligned_mols)}/{len(df)} compounds  "
          f"({len(failed)} skipped)")

    if len(aligned_mols) < 3:
        raise RuntimeError("Need at least 3 aligned molecules to build a model")

    # ── Build pharmacophore model ─────────────────────────────────────────────
    print("\nBuilding pharmacophore model...")
    model = build_pharmacophore(aligned_mols, factory)

    if not model:
        raise RuntimeError("No pharmacophore features survived the filters — "
                           "try lowering MIN_FEATURE_COVERAGE")

    # ── Print model ───────────────────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"PHARMACOPHORE MODEL  ({len(model)} features)")
    print(f"{'─'*70}")
    print(f"{'#':<4} {'Family':<12} {'X':>8} {'Y':>8} {'Z':>8} "
          f"{'Radius':>8} {'Coverage':>10}")
    print(f"{'─'*70}")
    for i, pt in enumerate(model, 1):
        cx, cy, cz = pt['center']
        print(f"{i:<4} {pt['family']:<12} {cx:>8.2f} {cy:>8.2f} {cz:>8.2f} "
              f"{pt['radius']:>8.2f} {pt['coverage']:>9.1%}")
    print(f"{'─'*70}")

    families_in_model = sorted({p['family'] for p in model})
    print(f"Feature families: {families_in_model}")

    # ── Score every molecule against the model ────────────────────────────────
    print(f"\n{'─'*70}")
    print("FIT SCORES")
    print(f"{'─'*70}")
    print(f"{'Name':<22} {'IC50 (µM)':>11} {'Fit Score':>10} {'Hits':>8}")
    print(f"{'─'*70}")

    results = []
    for mol in aligned_mols:
        name  = mol.GetProp("_Name") if mol.HasProp("_Name") else "unknown"
        pic50 = df.loc[df['Name'] == name, 'pIC50']
        pic50 = float(pic50.iloc[0]) if len(pic50) else float('nan')
        ic50  = 10 ** (6 - pic50) if not np.isnan(pic50) else float('nan')  # µM

        fit, hits = score_molecule(mol, model, factory)
        results.append({'name': name, 'ic50': ic50, 'fit': fit, 'hits': hits})

    # sort by fit score descending for easy reading
    for r in sorted(results, key=lambda x: -x['fit']):
        hit_str  = f"{r['hits']}/{len(model)}"
        ic50_str = f"{r['ic50']:.4f}" if not np.isnan(r['ic50']) else "N/A"
        print(f"{r['name']:<22} {ic50_str:>11} {r['fit']:>10.3f} {hit_str:>8}")

    print(f"{'─'*70}")

    # ── Save pharmacophore model as XML ──────────────────────────────────────
    xml_path = "pharmacophore_model.xml"
    save_model_xml(model, xml_path)
    print(f"\nPharmacophore model saved to: {xml_path}")

    print("\nDone.")


if __name__ == "__main__":
    run()

Mtb Q-Loop Pharmacophore Model (simple)
Feature factory built.
Compounds loaded: 22  (pIC50 4.80 – 8.52)
Template: CK_2_63.sdf

Aligning compounds to template...
  Aurachin D              RMSD=0.85 Å
  CK-3-22 (1T)            RMSD=0.63 Å
  CK-3-14                 RMSD=0.03 Å
  RKA-259                 NOT FOUND
  RKA-307                 RMSD=0.00 Å
  RKA-310                 RMSD=0.00 Å
  MTD-403                 RMSD=0.01 Å
  CK-2-88                 RMSD=0.00 Å
  CK-3-23                 RMSD=0.04 Å
  CK-2-63                 RMSD=0.00 Å
  PG-203                  RMSD=1.03 Å
  RKA-70                  RMSD=0.17 Å
  RKA-73                  RMSD=0.11 Å
  LT-9                    RMSD=0.70 Å
  GN-171                  RMSD=0.02 Å
  PG-128                  RMSD=0.54 Å
  SL-2-25                 RMSD=2.98 Å
  WDH-1U-10               RMSD=0.02 Å
  WDH-1W-5                NOT FOUND
  WDH-2A-9                RMSD=2.48 Å
  WDH-2G-6                RMSD=2.02 Å
  WDH-2R-4                RMSD=2.48 Å

Align